# CO₂ Emissions Analysis – Extended Portfolio
**Dataset**: Our World in Data CO₂ Emissions (1750–2023)  
**Sections**: Data Preparation → EDA → Visualisation → Extended Analysis (CO2_Analysis.md)


## 0. Imports & Global Settings

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Global style
sns.set_theme(style="whitegrid", palette="tab10", font_scale=1.0)
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FAFAFA',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

CONTINENT_NAMES = ['Africa', 'Asia', 'Europe', 'North America',
                   'South America', 'Oceania', 'Antarctica']
CONT_PALETTE = {
    'Africa':        '#E67E22',
    'Asia':          '#E74C3C',
    'Europe':        '#3498DB',
    'North America': '#2ECC71',
    'South America': '#9B59B6',
    'Oceania':       '#1ABC9C',
}


## 1. Data Preparation

In [ ]:
# Load raw dataset
df = pd.read_csv('owid-co2-data.csv')
print(f"Raw shape: {df.shape}")
print(f"Years: {df['year'].min()} – {df['year'].max()}")
print(f"Distinct entities: {df['country'].nunique()}")


In [ ]:
# Inspect missing values
columns_with_nas = df.columns[df.isna().any()]
nas_sum = df[columns_with_nas].isna().sum().sort_values(ascending=False)
print("Top 20 columns by missing value count:")
display(nas_sum.head(20))


In [ ]:
# Key descriptive statistics
print("Mean average from key metrics:")
for col in ['population', 'co2', 'methane', 'oil_co2', 'nitrous_oxide']:
    val = df[col].mean(skipna=True)
    print(f"  {col}: {val:,.2f}")

print()
print("Median from key metrics:")
for col in ['ghg_per_capita', 'primary_energy_consumption',
            'energy_per_capita', 'co2_per_capita', 'gdp']:
    val = df[col].median(skipna=True)
    print(f"  {col}: {val:,.2f}")

print()
print("Range from key metrics:")
for col in ['population', 'temperature_change_from_co2',
            'temperature_change_from_ghg', 'temperature_change_from_n2o',
            'temperature_change_from_ch4']:
    r = df[col].max() - df[col].min()
    print(f"  {col}: {r:,.4f}")


In [ ]:
# Tag continent rows vs country rows
df['is_continent'] = df['country'].isin(CONTINENT_NAMES)

# Filter: post-1990, non-null CO2
df_recent     = df[(df['year'] >= 1990) & df['co2'].notna()].copy()
df_countries  = df_recent[~df_recent['is_continent']].copy()
df_continents = df_recent[df_recent['is_continent']].copy()
df_cont_no_ant= df_continents[df_continents['country'] != 'Antarctica'].copy()

# Impute missing population with column median
pop_median = df_countries['population'].median()
df_countries['population'] = df_countries['population'].fillna(pop_median)
print(f"Population NAs imputed with median: {pop_median:,.0f}")

# Calculated CO2 per capita (tonnes per person)
df_countries['co2_per_capita_calc'] = np.where(
    df_countries['population'] > 0,
    df_countries['co2'] / df_countries['population'] * 1e6,
    np.nan
)

print(f"Post-1990 country rows: {len(df_countries):,}")
print(f"Countries: {df_countries['country'].nunique()}")
display(df_countries.head())


In [ ]:
# Convenience subset: 'pure' countries (exclude regional aggregates)
PURE_COUNTRIES = df_countries[~df_countries['country'].str.contains(
    'GCP|OECD|income|World|Union|G20|EU|Non-', na=False)].copy()
print(f"Pure country rows: {len(PURE_COUNTRIES):,}")


## 2. Exploratory Data Analysis

### 2.1 Data Coverage & Completeness

In [ ]:
# Countries reporting per year
coverage_by_year = df.groupby('year')['country'].nunique()
print(f"Peak country count: {coverage_by_year.max()} (year {coverage_by_year.idxmax()})")

# Countries with 30+ years of post-1990 data
country_years = df_countries.groupby('country')['year'].count()
long_record = country_years[country_years >= 30].sort_values(ascending=False)
print(f"Countries with 30+ years of post-1990 data: {len(long_record)}")
display(long_record.head(10))


### 2.2 Distribution of Emissions

In [ ]:
# Top 10 country-level share of global CO2 in 2020
df_2020 = PURE_COUNTRIES[PURE_COUNTRIES['year'] == 2020].copy()
total_2020 = df_2020['co2'].sum()
top10_2020 = df_2020.nlargest(10, 'co2')
top10_share = top10_2020['co2'].sum() / total_2020 * 100
print(f"Top 10 countries' share of global CO2 in 2020: {top10_share:.1f}%")
display(top10_2020[['country', 'co2', 'co2_per_capita']].reset_index(drop=True))


### 2.3 Historical Milestones

In [ ]:
df_world = df[df['country'] == 'World'].copy().sort_values('year')

for threshold in [1_000, 10_000, 30_000, 37_000]:
    year_hit = df_world[df_world['co2'] >= threshold]['year'].min()
    print(f"Global CO2 first exceeded {threshold:,} Mt in: {year_hit}")

df_world['yoy_growth'] = df_world['co2'].pct_change() * 100
print()
print("Fastest YoY global growth years:")
display(df_world.nlargest(5, 'yoy_growth')[['year', 'co2', 'yoy_growth']].reset_index(drop=True))


### 2.4 Summary Statistics by Continent

In [ ]:
# Mean & median CO2 emissions by continent (post-1990)
cont_stats = (df_cont_no_ant
              .groupby('country')['co2']
              .agg(mean_co2='mean', median_co2='median')
              .sort_values('mean_co2', ascending=False))
display(cont_stats)


### 2.5 Top 5 Emitting Countries in 2020

In [ ]:
top5 = PURE_COUNTRIES[PURE_COUNTRIES['year'] == 2020].nlargest(5, 'co2')
display(top5[['country', 'co2', 'co2_per_capita', 'gdp']].reset_index(drop=True))


### 2.6 COVID-19 Impact (2019 → 2020)

In [ ]:
df_2019 = PURE_COUNTRIES[PURE_COUNTRIES['year'] == 2019][['country', 'co2']].rename(columns={'co2': 'co2_2019'})
df_2020c= PURE_COUNTRIES[PURE_COUNTRIES['year'] == 2020][['country', 'co2']].rename(columns={'co2': 'co2_2020'})
covid = df_2019.merge(df_2020c, on='country')
covid['change_pct'] = (covid['co2_2020'] - covid['co2_2019']) / covid['co2_2019'] * 100

reduced = covid[covid['change_pct'] < 0]
print(f"Countries that reduced emissions in 2020: {len(reduced)} / {len(covid)}")
print()
print("Biggest drops:")
display(reduced.nsmallest(10, 'change_pct')[['country', 'change_pct']].reset_index(drop=True))


### 2.7 Cumulative Emissions (All-Time)

In [ ]:
df_hist = df[~df['is_continent'] & df['co2'].notna() &
             ~df['country'].str.contains('GCP|OECD|income|Union|G20|EU|Non-', na=False)]
cumul = df_hist.groupby('country')['co2'].sum().sort_values(ascending=False)
print("Top 10 cumulative emitters since 1750:")
display(cumul.head(10).to_frame())


### 2.8 GDP–CO₂ Correlation & Decoupling

In [ ]:
df_gdp = df_countries.dropna(subset=['gdp', 'co2'])
corr = df_gdp[['gdp', 'co2']].corr().iloc[0, 1]
print(f"Overall Pearson correlation (GDP vs CO2, post-1990): {corr:.3f}")

# Countries with negative GDP–CO2 correlation since 2000 (decoupling)
def decoupling_test(g):
    if len(g) < 10: return np.nan
    if g['gdp'].isnull().mean() > 0.5: return np.nan
    g = g.dropna(subset=['gdp', 'co2'])
    return np.corrcoef(g['gdp'], g['co2'])[0, 1] if len(g) >= 5 else np.nan

decoupling = (PURE_COUNTRIES[PURE_COUNTRIES['year'] >= 2000]
              .groupby('country').apply(decoupling_test)
              .dropna().sort_values())
print()
print("Top 10 countries with strongest decoupling (negative GDP–CO2 correlation):")
display(decoupling.head(10).to_frame(name='gdp_co2_corr'))


### 2.9 Emission Intensity Trend

In [ ]:
df_intensity = df_world[df_world['year'] >= 1990].dropna(subset=['co2_per_gdp'])
i_1990 = df_intensity[df_intensity['year'] == 1990]['co2_per_gdp'].values[0]
i_2020 = df_intensity[df_intensity['year'] == 2020]['co2_per_gdp'].values[0]
print(f"World CO2/GDP in 1990: {i_1990:.4f} Mt per int$")
print(f"World CO2/GDP in 2020: {i_2020:.4f} Mt per int$")
print(f"Improvement: {(1 - i_2020/i_1990)*100:.1f}% reduction in carbon intensity")


### 2.10 Anomaly Detection

In [ ]:
df_sorted = PURE_COUNTRIES.sort_values(['country', 'year']).copy()
df_sorted['pct_change'] = df_sorted.groupby('country')['co2'].pct_change() * 100

print("Largest single-year % increases in CO2 (post-1990):")
display(df_sorted.nlargest(10, 'pct_change')[['country', 'year', 'co2', 'pct_change']].reset_index(drop=True))

negative = PURE_COUNTRIES[PURE_COUNTRIES['co2'] < 0]
print(f"\nRows with negative CO2 emissions: {len(negative)}")


### 2.11 Linear Projection

In [ ]:
df_proj = df_world[df_world['year'] >= 2010].dropna(subset=['co2'])
coeffs = np.polyfit(df_proj['year'], df_proj['co2'], 1)
target = 40_000
year_proj = (target - coeffs[1]) / coeffs[0]
avg_rate = df_world.dropna(subset=['co2'])['yoy_growth'].tail(20).mean()
print(f"Linear trend projects 40,000 Mt/yr by: ~{year_proj:.0f}")
print(f"Average annual global growth rate (last 20 years): {avg_rate:.2f}%")


## 3. Data Visualisation

### Chart 1 – Global CO₂ Emissions Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
ax.fill_between(df_world['year'], df_world['co2'], alpha=0.15, color='#E74C3C')
ax.plot(df_world['year'], df_world['co2'], color='#E74C3C', linewidth=2.2)

milestones = [(1884, 1_000, '1 Bt (1884)'),
              (1963, 10_000, '10 Bt (1963)'),
              (2006, 30_000, '30 Bt (2006)')]
for yr, val, label in milestones:
    ax.axhline(val, color='grey', linewidth=0.7, linestyle='--', alpha=0.5)
    ax.annotate(label, xy=(yr, val), xytext=(yr + 2, val + 500), fontsize=8, color='grey')

ax.set(title='Global CO₂ Emissions Over Time (1750–2023)',
       xlabel='Year', ylabel='CO₂ Emissions (million tonnes)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()


### Chart 2 – CO₂ by Continent (Line Chart, 1990–2023)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for cont, grp in df_cont_no_ant.groupby('country'):
    ax.plot(grp.sort_values('year')['year'], grp.sort_values('year')['co2'],
            label=cont, color=CONT_PALETTE.get(cont), linewidth=2)
ax.set(title='CO₂ Emissions by Continent (1990–2023)',
       xlabel='Year', ylabel='CO₂ Emissions (million tonnes)')
ax.legend(title='Continent', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()


### Chart 3 – Boxplot: CO₂ Per Capita by Continent

In [ ]:
order_by_median = (df_cont_no_ant.groupby('country')['co2_per_capita']
                   .median().sort_values(ascending=False).index.tolist())
palette = [CONT_PALETTE.get(c, '#aaa') for c in order_by_median]

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=df_cont_no_ant, x='country', y='co2_per_capita',
            order=order_by_median, palette=palette, ax=ax,
            flierprops=dict(marker='o', markersize=3, alpha=0.4))
ax.set(title='CO₂ Per Capita by Continent (1990–2023)',
       xlabel='Continent', ylabel='CO₂ Per Capita (tonnes/person)')
plt.tight_layout()
plt.show()


### Chart 4 – CO₂ vs GDP Scatter (with Regression Line)

In [ ]:
df_scatter = PURE_COUNTRIES[PURE_COUNTRIES['year'] == 2020].dropna(subset=['gdp', 'co2'])
df_scatter = df_scatter[df_scatter['co2'] > 0]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_scatter['gdp'] / 1e12, df_scatter['co2'],
           alpha=0.55, s=40, color='#3498DB', edgecolors='white', linewidth=0.4)

m, b = np.polyfit(df_scatter['gdp'], df_scatter['co2'], 1)
x_line = np.linspace(df_scatter['gdp'].min(), df_scatter['gdp'].max(), 200)
ax.plot(x_line / 1e12, m * x_line + b, color='#E74C3C', linewidth=1.8,
        linestyle='--', label='Regression line')

for _, row in df_scatter.nlargest(5, 'co2').iterrows():
    ax.annotate(row['country'], (row['gdp'] / 1e12, row['co2']),
                textcoords='offset points', xytext=(4, 2), fontsize=7.5)

ax.set(title='CO₂ Emissions vs GDP – Country Level (2020)',
       xlabel='GDP (Trillion USD, PPP)', ylabel='CO₂ Emissions (million tonnes)')
ax.legend()
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()


### Chart 5 – Top 5 Emitting Countries in 2020

In [ ]:
top5_2020 = PURE_COUNTRIES[PURE_COUNTRIES['year'] == 2020].nlargest(5, 'co2')

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(top5_2020['country'][::-1], top5_2020['co2'][::-1],
               color=['#E74C3C','#E67E22','#F1C40F','#2ECC71','#3498DB'])
for bar, val in zip(bars, top5_2020['co2'][::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f} Mt', va='center', fontsize=9)
ax.set(title='Top 5 CO₂ Emitting Countries in 2020',
       xlabel='CO₂ Emissions (million tonnes)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.tight_layout()
plt.show()


### Chart 6 – COVID-19 Impact on CO₂ (2019 → 2020)

In [ ]:
covid_top = (PURE_COUNTRIES[PURE_COUNTRIES['year'].isin([2019, 2020])]
             .groupby('country')['co2'].mean().nlargest(15).index)
covid_filtered = covid[covid['country'].isin(covid_top)].sort_values('change_pct')

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#E74C3C' if x < 0 else '#2ECC71' for x in covid_filtered['change_pct']]
ax.barh(covid_filtered['country'], covid_filtered['change_pct'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set(title='COVID-19 Impact: % Change in CO₂ 2019→2020 (Top 15 Emitters)',
       xlabel='% Change in CO₂ Emissions')
plt.tight_layout()
plt.show()


### Chart 7 – Cumulative Emitters (All-Time)

In [ ]:
cumul_top10 = cumul.head(10).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cumul_top10.index, cumul_top10.values / 1e3,
               color=sns.color_palette("Reds_r", len(cumul_top10)))
for bar, val in zip(bars, cumul_top10.values / 1e3):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}', va='center', fontsize=8.5)
ax.set(title='Cumulative CO₂ Emissions Since 1750 – Top 10 Countries',
       xlabel='Cumulative CO₂ Emissions (billion tonnes)')
plt.tight_layout()
plt.show()


### Chart 8 – GDP Growth vs CO₂ Change (Decoupling)

In [ ]:
decoupling_countries = ['Sweden','Germany','United Kingdom','Denmark','France',
                        'Belgium','United States','Japan','China','India']
df_decouple = PURE_COUNTRIES[PURE_COUNTRIES['country'].isin(decoupling_countries)].copy()

latest = (df_decouple.dropna(subset=['gdp','co2'])
          .groupby('country')['year'].max().reset_index()
          .rename(columns={'year': 'latest_year'}))
earliest = (df_decouple[df_decouple['year'] == 2000]
            .dropna(subset=['gdp','co2'])[['country','gdp','co2']]
            .rename(columns={'gdp': 'gdp_2000', 'co2': 'co2_2000'}))

pivoted = latest.merge(earliest, on='country')
def get_latest_vals(row):
    r = df_decouple[(df_decouple['country'] == row['country']) &
                    (df_decouple['year'] == row['latest_year'])].iloc[0]
    return pd.Series({'gdp_latest': r['gdp'], 'co2_latest': r['co2']})

pivoted = pivoted.join(pivoted.apply(get_latest_vals, axis=1)).dropna()
pivoted['gdp_change'] = (pivoted['gdp_latest'] - pivoted['gdp_2000']) / pivoted['gdp_2000'] * 100
pivoted['co2_change'] = (pivoted['co2_latest'] - pivoted['co2_2000']) / pivoted['co2_2000'] * 100

fig, ax = plt.subplots(figsize=(10, 6))
colors_d = ['#E74C3C' if co2 > 0 else '#2ECC71' for co2 in pivoted['co2_change']]
ax.scatter(pivoted['gdp_change'], pivoted['co2_change'], s=120,
           c=colors_d, edgecolors='white', linewidth=0.8, zorder=3)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.fill_betweenx([-60, 0], [0, 0], [600, 600], alpha=0.06, color='green')
ax.text(200, -40, 'Decoupled\n(GDP ↑, CO₂ ↓)', color='green', fontsize=9, alpha=0.8)
for _, row in pivoted.iterrows():
    ax.annotate(row['country'], (row['gdp_change'], row['co2_change']),
                textcoords='offset points', xytext=(5, 3), fontsize=8)
ax.set(title='GDP Growth vs CO₂ Change (2000 to Latest Year)\nGreen quadrant = decoupling achieved',
       xlabel='GDP Growth (%)', ylabel='CO₂ Change (%)')
plt.tight_layout()
plt.show()


### Chart 9 – Global Carbon Intensity of GDP

In [ ]:
df_intensity = df_world[df_world['year'] >= 1990].dropna(subset=['co2_per_gdp'])

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_intensity['year'], df_intensity['co2_per_gdp'] * 1000,
        color='#9B59B6', linewidth=2.2)
ax.fill_between(df_intensity['year'], df_intensity['co2_per_gdp'] * 1000,
                alpha=0.12, color='#9B59B6')
ax.set(title='Global Carbon Intensity of GDP (1990–2023)',
       xlabel='Year', ylabel='CO₂ per $1,000 GDP (kg/$)')
plt.tight_layout()
plt.show()
